# Resolving a Lookahead Conflict

The notebook [03-Look-Ahead.ipynb](03-Look-Ahead.ipynb) has discussed the grammar
```
    a : b "U" "V"
      | c "U" "W"

    b : "X"

    c : "X"
```
which is *unambiguous* but nevertheless not an `LR(1)` grammar:  after an `'X'` has been read and the next
token is a `'U'`, the parser cannot decide whether to reduce with the rule for `b` or with the rule for `c`,
since that is only settled by the token *following* the `'U'`.  `Larkita` therefore refuses to build an `LR(1)`
parser and raises a `GrammarError`.

The problem is that the grammar tries to interpret an `'X'` either as a `b` or as a `c` depending on the
context.  The remedy is to pull the context `'U'`, which follows both `b` and `c`, into the rules for `b`
and `c`:
```
    a : b "V"
      | c "W"

    b : "X" "U"

    c : "X" "U"
```
This grammar generates the same language, but now the decision between `b` and `c` is postponed until the
decisive token is the *next* token.  We will see that there are no conflicts.

## Specification of the Grammar

In [ ]:
from IPython.display import Code

Code(filename='look-ahead-solved.lark', language='larkita')

## Specification of the Parser

We turn the `ConflictWarning` that `Larkita` issues for a grammar with shift/reduce conflicts into an exception.
A reduce/reduce conflict would raise a `GrammarError` anyway.  Since **no** exception is raised by the cell below, the
grammar has no conflicts and is therefore an `LR(1)` grammar.

The rules for `b` and `c` have the same literals `"X"` and `"U"`, which would form the heads of their abstract syntax
trees.  The *aliases* `-> b` and `-> c` replace these heads, so that we can see in the abstract syntax trees at the
end of this notebook which of the two rules has been used.

In [ ]:
import warnings

from larkita import Larkita, ConflictWarning

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter('error', ConflictWarning)
    parser = Larkita.open('look-ahead-solved.lark', out_file='look-ahead-solved.out')

## Inspecting the LR(1) States

`Larkita` has written the states of the canonical LR(1) automaton into the file `look-ahead-solved.out`.  Every state
lists its *marked rules* together with their *lookahead* tokens, followed by the actions and the gotos of the state.

In [ ]:
with open('look-ahead-solved.out') as handle:
    print(handle.read())

The interesting state is the one that contains the two marked rules
```
    (3)  b → "X" "U" •  { "V" }
    (4)  c → "X" "U" •  { "W" }
```
This is the state that corresponds to the state which caused the conflict in the previous notebook.  In the
notation of the lecture notes it has the form
$$ \{\; b \rightarrow \texttt{'X'}\,\texttt{'U'} \bullet : \texttt{'V'}, \quad
       c \rightarrow \texttt{'X'}\,\texttt{'U'} \bullet : \texttt{'W'} \;\} $$
and the follow sets of the two rules are now *disjoint*.  Therefore the action table of this state offers
exactly one reduce action per token:  a `'V'` reduces with the rule for `b`, while a `'W'` reduces with the
rule for `c`.  No other state contains more than one marked rule whose position is at the very end, so there
is no conflict anywhere.

## Parsing Some Strings

Finally, we check that the parser really distinguishes a `b` from a `c`.

In [ ]:
parser.parse('XUV')

In [ ]:
parser.parse('XUW')

A string that is not derivable from `a` is rejected.  There is no need for a function like `p_error`:
`Larkita` raises an exception of class `UnexpectedInput` whose message already reports the position and the
tokens that would have been acceptable.

In [ ]:
from larkita import UnexpectedInput

try:
    parser.parse('XUZ')
except UnexpectedInput as e:
    print(e)